In [1]:
import climakitae as ck
from climakitae.core.data_interface import get_data
import xarray as xr
import geopandas as gpd
import warnings
import os
import gc
import time

# --- Configuration ---
warnings.filterwarnings("ignore")

STANDARD_CRS = "EPSG:4326"
DOWNSCALING = "Statistical"
RESOLUTION = "3 km"

# Variable Mapping
VARIABLES_STAT = {
    "T_Max": "Maximum air temperature at 2m",
    "T_Min": "Minimum air temperature at 2m",
    "Precip": "Precipitation (total)"
}

SCENARIOS = ["Historical Climate", "SSP 3-7.0"]

# Output Directory
output_dir = "dataForRScripts/NetCDF_Chunks"
os.makedirs(output_dir, exist_ok=True)

# Boundaries
SHAPEFILES = {
    "JoshuaTree": "../JoshuaTreeOutlines/JoshuaTree/Joshua_Tree_National_Park.shp",
    "Mojave": "../Mojave/Mojave_National_Preserve.shp"
}

BOUNDARIES = {}
for name, path in SHAPEFILES.items():
    try:
        BOUNDARIES[name] = gpd.read_file(path).to_crs(STANDARD_CRS)
    except:
        pass

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

# --- Main Loop ---
log("=== Starting Fast NetCDF Export ===")

for region, gdf in BOUNDARIES.items():
    if gdf is None: continue
    log(f"Processing {region}...")

    bounds = gdf.total_bounds
    # Pad bounds slightly to ensure we don't clip too tight before saving
    lat_slice = (bounds[1] - 0.05, bounds[3] + 0.05)
    lon_slice = (bounds[0] - 0.05, bounds[2] + 0.05)

    for var_short, var_full in VARIABLES_STAT.items():
        log(f"  Fetching {var_short}...")
        
        # Process in 10-year chunks
        for start_year in range(1970, 2100, 10):
            end_year = min(start_year + 9, 2099)
            
            # File check: Skip if already exists
            filename = f"{region}_{var_short}_{start_year}_{end_year}.nc"
            filepath = os.path.join(output_dir, filename)
            if os.path.exists(filepath):
                continue
            
            try:
                # 1. Fetch
                ds = get_data(
                    variable=var_full, resolution=RESOLUTION, downscaling_method=DOWNSCALING,
                    timescale="monthly", scenario=SCENARIOS, 
                    time_slice=(start_year, end_year),
                    latitude=lat_slice, longitude=lon_slice
                )
                
                # 2. Preprocess
                if 'units' in ds.attrs and ds.attrs['units'] == 'K': 
                    ds = ds - 273.15
                    ds.attrs['units'] = 'C'
                
                if var_short == 'Precip' and ds.attrs.get('units') in ['kg/m^2/s', 'kg m-2 s-1']:
                    days = ds.time.dt.days_in_month
                    ds = (ds * 86400) * days
                    ds.attrs['units'] = 'mm/month'

                # 3. Clip
                if ds.rio.crs is None: ds.rio.write_crs(STANDARD_CRS, inplace=True)
                ds = ds.rio.clip(gdf.geometry.values, drop=True)

                # 4. Save
                # We save the monthly data directly. We can Average in R later.
                # This is faster than computing averages here.
                ds.to_netcdf(filepath)
                log(f"    Saved {filename}")

                del ds
                gc.collect()

            except Exception as e:
                log(f"    Error {start_year}-{end_year}: {e}")

log("=== Done ===")

[20:19:38] === Starting Fast NetCDF Export ===
[20:19:38] Processing JoshuaTree...
[20:19:38]   Fetching T_Max...
[20:19:38]   Fetching T_Min...


KeyboardInterrupt: 